# Lab 05: Deploy and Inspect a Governed Agent

**Day 1 - Session 5**

Goal: Operate a pre-configured AgentCore application that uses a Salesforce service-case scenario and a retail order-status tool.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/labs/05-operating-the-environment/start/05-operating-the-environment.ipynb)

In [ ]:
# Run this cell once to install required packages.
!pip install "boto3>=1.43.62" --quiet

## Configuration

You must set your environment variables as provided by the instructor.

In [1]:
import os

os.environ["AWS_REGION"] = input("AWS_REGION (e.g. us-east-1): ").strip()
os.environ["AGENT_NAME"] = input("AGENT_NAME (e.g. sf_case_team01_am): ").strip()
os.environ["S3_KEY"] = input("S3_KEY: ").strip()
os.environ["GATEWAY_URL"] = input("GATEWAY_URL: ").strip()
os.environ["GATEWAY_TOOL_NAME"] = input("GATEWAY_TOOL_NAME: ").strip()
os.environ["MEMORY_ID"] = input("MEMORY_ID: ").strip()
os.environ["ACTOR_ID"] = input("ACTOR_ID: ").strip()
os.environ["MEMORY_SESSION_ID"] = input("MEMORY_SESSION_ID: ").strip()
os.environ["ORDER_ID"] = input("ORDER_ID: ").strip()
os.environ["EXPECTED_ORDER_STATUS"] = input("EXPECTED_ORDER_STATUS: ").strip()

# DO NOT reset EXECUTION_ROLE_ARN or S3_BUCKET if they are already in the environment
if "EXECUTION_ROLE_ARN" not in os.environ:
    os.environ["EXECUTION_ROLE_ARN"] = input("EXECUTION_ROLE_ARN: ").strip()
if "S3_BUCKET" not in os.environ:
    os.environ["S3_BUCKET"] = input("S3_BUCKET: ").strip()

print("Configuration loaded.")

Configuration loaded.


## Review `agent.py`

This is the source code deployed inside the AgentCore Runtime. **Read only, do not run.** It starts an HTTP server meant to run inside the managed microVM.

In [ ]:
%%writefile agent.py
"""
Lab 05 - Governed agent for the Salesforce case + retail order-status scenario.

Deployed to AgentCore Runtime as a direct-code Python ZIP (entryPoint agent.py).
Implements the Runtime HTTP contract directly (no bedrock-agentcore SDK):
  GET  /ping          health check
  POST /invocations   {"prompt": ..., "actorId": ..., "memorySessionId": ...}

On each invocation the agent:
  1. Loads prior conversation turns for this actorId/memorySessionId from
     AgentCore Memory, so a second, different Runtime session can recall
     what an earlier session was told.
  2. Asks the model to answer the prompt, offering the read-only Gateway
     order-status tool.
  3. Calls the Gateway over MCP (SigV4-signed with the Runtime's own
     execution role) whenever the model asks for the tool.
  4. Writes both the user prompt and the final answer back to Memory.
"""
import json
import os
import sys
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import urllib.request

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
GATEWAY_URL = os.environ.get("GATEWAY_URL", "")
GATEWAY_TOOL_NAME = os.environ.get("GATEWAY_TOOL_NAME", "")
MEMORY_ID = os.environ.get("MEMORY_ID", "")
MODEL_ID = os.environ.get("MODEL_ID", "amazon.nova-lite-v1:0")

SYSTEM_PROMPT = (
    "You help with two kinds of requests: checking a retail order's status "
    "using the order_status tool, and remembering short facts the user tells "
    "you (such as a Salesforce case number) so you can recall them later in "
    "a different conversation. Always call order_status instead of guessing "
    "a status. When asked to recall something, answer only from the "
    "conversation history provided to you."
)

TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "order_status",
                "description": "Look up the status of a retail order.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "orderId": {
                                "type": "string",
                                "description": "The order identifier to look up, e.g. ORD-1001",
                            }
                        },
                        "required": ["orderId"],
                    }
                },
            }
        }
    ]
}

_bedrock = None
_agentcore = None


def bedrock_client():
    global _bedrock
    if _bedrock is None:
        _bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
    return _bedrock


def agentcore_client():
    global _agentcore
    if _agentcore is None:
        _agentcore = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
    return _agentcore


def call_gateway_tool(order_id: str) -> str:
    """Invoke the Gateway's order-status MCP tool with the Runtime's own SigV4 identity."""
    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "tools/call",
        "params": {"name": GATEWAY_TOOL_NAME, "arguments": {"orderId": order_id}},
    }
    body = json.dumps(payload).encode("utf-8")
    credentials = boto3.Session(region_name=AWS_REGION).get_credentials()
    request = AWSRequest(
        method="POST",
        url=GATEWAY_URL,
        data=body,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "MCP-Protocol-Version": "2025-03-26",
        },
    )
    SigV4Auth(credentials.get_frozen_credentials(), "bedrock-agentcore", AWS_REGION).add_auth(request)
    urllib_request = urllib.request.Request(
        GATEWAY_URL, data=body, headers=dict(request.headers.items()), method="POST"
    )
    with urllib.request.urlopen(urllib_request, timeout=20) as response:
        result = json.loads(response.read().decode("utf-8"))
    if "error" in result:
        return json.dumps({"error": result["error"]})
    content = result.get("result", {}).get("content", [])
    texts = [item["text"] for item in content if item.get("type") == "text"]
    return "\n".join(texts) if texts else json.dumps(result.get("result", {}))


def load_memory_turns(actor_id: str, session_id: str):
    """Return prior conversation turns for this actor/session as Converse messages."""
    if not MEMORY_ID:
        return []
    response = agentcore_client().list_events(
        memoryId=MEMORY_ID,
        actorId=actor_id,
        sessionId=session_id,
        includePayloads=True,
        maxResults=100,
    )
    events = sorted(response.get("events", []), key=lambda e: e.get("eventTimestamp", 0))
    messages = []
    for event in events:
        for item in event.get("payload", []):
            turn = item.get("conversational")
            if not turn:
                continue
            text = turn.get("content", {}).get("text", "")
            role = turn.get("role", "USER")
            if not text:
                continue
            messages.append({"role": "user" if role == "USER" else "assistant", "content": [{"text": text}]})
    return messages


def write_memory_turn(actor_id: str, session_id: str, role: str, text: str) -> None:
    if not MEMORY_ID:
        return
    agentcore_client().create_event(
        memoryId=MEMORY_ID,
        actorId=actor_id,
        sessionId=session_id,
        eventTimestamp=time.time(),
        payload=[{"conversational": {"content": {"text": text}, "role": role}}],
    )


def run_conversation(prompt: str, history: list) -> str:
    messages = history + [{"role": "user", "content": [{"text": prompt}]}]

    for _ in range(4):
        response = bedrock_client().converse(
            modelId=MODEL_ID,
            system=[{"text": SYSTEM_PROMPT}],
            messages=messages,
            toolConfig=TOOL_CONFIG,
        )
        output_message = response["output"]["message"]
        messages.append(output_message)

        if response.get("stopReason") != "tool_use":
            return "".join(block.get("text", "") for block in output_message["content"])

        tool_results = []
        for block in output_message["content"]:
            tool_use = block.get("toolUse")
            if not tool_use:
                continue
            if tool_use["name"] == "order_status":
                result_text = call_gateway_tool(tool_use["input"]["orderId"])
            else:
                result_text = json.dumps({"error": f"unknown tool {tool_use['name']}"})
            tool_results.append(
                {
                    "toolResult": {
                        "toolUseId": tool_use["toolUseId"],
                        "content": [{"text": result_text}],
                    }
                }
            )
        messages.append({"role": "user", "content": tool_results})

    return "I could not complete that request."


def handle_invocation(body: dict) -> dict:
    prompt = body.get("prompt", "")
    actor_id = body.get("actorId", "")
    session_id = body.get("memorySessionId", "")

    history = load_memory_turns(actor_id, session_id)
    answer = run_conversation(prompt, history)

    write_memory_turn(actor_id, session_id, "USER", prompt)
    write_memory_turn(actor_id, session_id, "ASSISTANT", answer)

    return {"response": answer}


class Handler(BaseHTTPRequestHandler):
    def _write_json(self, status: int, body: dict) -> None:
        payload = json.dumps(body).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def do_GET(self):
        if self.path == "/ping":
            self._write_json(200, {"status": "Healthy"})
            return
        self._write_json(404, {"error": "not found"})

    def do_POST(self):
        if self.path != "/invocations":
            self._write_json(404, {"error": "not found"})
            return
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length) if length else b"{}"
        try:
            body = json.loads(raw.decode("utf-8"))
            result = handle_invocation(body)
            self._write_json(200, result)
        except Exception as e:
            print(f"INVOCATION_ERROR: {e}", file=sys.stderr, flush=True)
            self._write_json(500, {"error": str(e)})

    def log_message(self, format, *args):
        print(f"{self.address_string()} - {format % args}", flush=True)


def main():
    print("AGENT_STARTING: binding 0.0.0.0:8080", flush=True)
    server = ThreadingHTTPServer(("0.0.0.0", 8080), Handler)
    print("AGENT_READY: listening on 0.0.0.0:8080", flush=True)
    server.serve_forever()


if __name__ == "__main__":
    main()


## Execute `deploy_agent.py`

Run the cell below.

In [ ]:
"""
Lab 05 - Deploy or update an AgentCore Runtime.

Requires a prebuilt direct-deployment ZIP in the assigned S3 bucket.
Run: python3 deploy_agent.py
"""
import os
import re
import time
import uuid

import boto3
from botocore.exceptions import ClientError

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
AGENT_NAME = os.environ.get("AGENT_NAME", "")
AGENT_RUNTIME_ID = os.environ.get("AGENT_RUNTIME_ID", "")
EXECUTION_ROLE_ARN = os.environ.get("EXECUTION_ROLE_ARN", "")
S3_BUCKET = os.environ.get("S3_BUCKET", "")
S3_KEY = os.environ.get("S3_KEY", "")
GATEWAY_URL = os.environ.get("GATEWAY_URL", "")
GATEWAY_TOOL_NAME = os.environ.get("GATEWAY_TOOL_NAME", "")
MEMORY_ID = os.environ.get("MEMORY_ID", "")


def require_configuration():
    required = {
        "AGENT_NAME": AGENT_NAME,
        "EXECUTION_ROLE_ARN": EXECUTION_ROLE_ARN,
        "S3_BUCKET": S3_BUCKET,
        "S3_KEY": S3_KEY,
        "GATEWAY_URL": GATEWAY_URL,
        "GATEWAY_TOOL_NAME": GATEWAY_TOOL_NAME,
        "MEMORY_ID": MEMORY_ID,
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise SystemExit(f"CONFIG_MISSING: export {', '.join(missing)}")
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,47}", AGENT_NAME):
        raise SystemExit("CONFIG_INVALID: AGENT_NAME must start with a letter and contain only letters, digits, or underscores")
    if not S3_KEY.endswith(".zip"):
        raise SystemExit("CONFIG_INVALID: S3_KEY must identify the direct-deployment ZIP object")


def runtime_configuration():
    return {
        "agentRuntimeArtifact": {
            "codeConfiguration": {
                "code": {"s3": {"bucket": S3_BUCKET, "prefix": S3_KEY}},
                "runtime": "PYTHON_3_12",
                "entryPoint": ["agent.py"],
            }
        },
        "roleArn": EXECUTION_ROLE_ARN,
        "networkConfiguration": {"networkMode": "PUBLIC"},
        "description": f"AgentCore course Runtime for {AGENT_NAME}",
        "lifecycleConfiguration": {
            "idleRuntimeSessionTimeout": 300,
            "maxLifetime": 28800,
        },
        "environmentVariables": {
            "COURSE_SCENARIO": "salesforce-case-and-retail-order",
            "AWS_REGION": AWS_REGION,
            "GATEWAY_URL": GATEWAY_URL,
            "GATEWAY_TOOL_NAME": GATEWAY_TOOL_NAME,
            "MEMORY_ID": MEMORY_ID,
        },
    }


def find_runtime_by_name(control, name: str):
    """Return the existing Runtime dict with this name, or None."""
    paginator = control.get_paginator("list_agent_runtimes")
    for page in paginator.paginate():
        for runtime in page.get("agentRuntimes", []):
            if runtime.get("agentRuntimeName") == name:
                return runtime
    return None


def wait_until_ready(control, runtime_id: str, version: str):
    deadline = time.time() + 300
    while time.time() < deadline:
        runtime = control.get_agent_runtime(
            agentRuntimeId=runtime_id,
            agentRuntimeVersion=version,
        )
        status = runtime["status"]
        print(f"DEPLOYMENT_WAIT: status={status}")
        if status == "READY":
            return runtime
        if status in {"CREATE_FAILED", "UPDATE_FAILED"}:
            raise SystemExit(f"DEPLOYMENT_FAILED: {runtime.get('failureReason', status)}")
        time.sleep(10)
    raise SystemExit("DEPLOYMENT_TIMEOUT: Runtime did not become READY within five minutes")


def create_runtime(control, configuration):
    print(f"DEPLOYMENT_START: creating Runtime {AGENT_NAME}")
    return control.create_agent_runtime(
        agentRuntimeName=AGENT_NAME,
        clientToken=str(uuid.uuid4()),
        **configuration,
    )


def wait_until_deleted(control, runtime_id: str, name: str):
    deadline = time.time() + 120
    while time.time() < deadline:
        if find_runtime_by_name(control, name) is None:
            return
        print(f"DEPLOYMENT_WAIT: waiting for Runtime {runtime_id} to finish deleting")
        time.sleep(5)
    raise SystemExit(
        f"DEPLOYMENT_TIMEOUT: Runtime {runtime_id} did not finish deleting within two minutes"
    )


def deploy():
    require_configuration()
    session = boto3.Session(region_name=AWS_REGION)
    session.client("s3").head_object(Bucket=S3_BUCKET, Key=S3_KEY)
    control = session.client("bedrock-agentcore-control")
    configuration = runtime_configuration()

    runtime_id = AGENT_RUNTIME_ID
    if not runtime_id:
        existing = find_runtime_by_name(control, AGENT_NAME)
        if existing:
            runtime_id = existing["agentRuntimeId"]
            print(f"DEPLOYMENT_INFO: found existing Runtime {runtime_id} named {AGENT_NAME}")

    if runtime_id:
        print(f"DEPLOYMENT_START: updating Runtime {runtime_id}")
        response = control.update_agent_runtime(
            agentRuntimeId=runtime_id,
            **configuration,
        )
    else:
        try:
            response = create_runtime(control, configuration)
        except ClientError as e:
            if e.response["Error"]["Code"] != "ConflictException":
                raise
            # Name collision with no discoverable match (e.g. propagation lag,
            # or a Runtime left over from an interrupted run) — clear it and
            # retry once rather than forcing a manual delete every time.
            conflicting = find_runtime_by_name(control, AGENT_NAME)
            if not conflicting:
                raise
            print(
                f"DEPLOYMENT_INFO: deleting conflicting Runtime "
                f"{conflicting['agentRuntimeId']} named {AGENT_NAME}"
            )
            control.delete_agent_runtime(agentRuntimeId=conflicting["agentRuntimeId"])
            wait_until_deleted(control, conflicting["agentRuntimeId"], AGENT_NAME)
            response = create_runtime(control, configuration)

    runtime = wait_until_ready(
        control,
        response["agentRuntimeId"],
        response["agentRuntimeVersion"],
    )
    print("DEPLOYMENT_OK")
    print(f"AGENT_RUNTIME_ID={runtime['agentRuntimeId']}")
    print(f"AGENT_RUNTIME_VERSION={runtime['agentRuntimeVersion']}")
    print(f"AGENT_RUNTIME_ARN={runtime['agentRuntimeArn']}")
    write_deploy_env(runtime)
    return runtime


def write_deploy_env(runtime):
    """
    Write deploy.env with this run's Runtime identifiers so later checkpoints
    (verify_memory.py, etc.) can `source deploy.env` instead of copy-pasting
    values printed to the terminal.
    """
    path = os.path.join(os.getcwd(), "deploy.env")
    with open(path, "w") as f:
        f.write(f"export AGENT_RUNTIME_ID={runtime['agentRuntimeId']}\n")
        f.write(f"export AGENT_RUNTIME_VERSION={runtime['agentRuntimeVersion']}\n")
        f.write(f"export AGENT_RUNTIME_ARN={runtime['agentRuntimeArn']}\n")
    print(f"DEPLOYMENT_INFO: wrote {path} — run `source deploy.env` to load these into your shell")


if __name__ == "__main__":
    deploy()


## Execute `check_gateway.py`

Run the cell below.

In [ ]:
"""
Lab 05 - Invoke a pre-created AgentCore Gateway tool with IAM SigV4.

Run: python3 check_gateway.py
"""
import json
import os
import urllib.error
import urllib.request
import uuid

import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
GATEWAY_URL = os.environ.get("GATEWAY_URL", "")
GATEWAY_TOOL_NAME = os.environ.get("GATEWAY_TOOL_NAME", "")
ORDER_ID = os.environ.get("ORDER_ID", "ORD-1001")
EXPECTED_ORDER_STATUS = os.environ.get("EXPECTED_ORDER_STATUS", "SHIPPED")


def require_configuration():
    missing = [name for name, value in {
        "GATEWAY_URL": GATEWAY_URL,
        "GATEWAY_TOOL_NAME": GATEWAY_TOOL_NAME,
    }.items() if not value]
    if missing:
        raise SystemExit(f"CONFIG_MISSING: export {', '.join(missing)}")
    if not GATEWAY_URL.startswith("https://"):
        raise SystemExit("CONFIG_INVALID: GATEWAY_URL must use HTTPS")


def signed_headers(body: bytes):
    credentials = boto3.Session(region_name=AWS_REGION).get_credentials()
    if credentials is None:
        raise SystemExit("IAM_FAILED: no AWS credentials are available")
    request = AWSRequest(
        method="POST",
        url=GATEWAY_URL,
        data=body,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "MCP-Protocol-Version": "2025-03-26",
        },
    )
    SigV4Auth(credentials.get_frozen_credentials(), "bedrock-agentcore", AWS_REGION).add_auth(request)
    return dict(request.headers.items())


def main():
    require_configuration()
    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "tools/call",
        "params": {
            "name": GATEWAY_TOOL_NAME,
            "arguments": {"orderId": ORDER_ID},
        },
    }
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        GATEWAY_URL,
        data=body,
        headers=signed_headers(body),
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise SystemExit(f"GATEWAY_FAILED: HTTP {error.code}: {detail}") from error

    if result.get("id") != payload["id"]:
        raise SystemExit("GATEWAY_FAILED: JSON-RPC response ID did not match the request")
    if "error" in result:
        raise SystemExit(f"GATEWAY_FAILED: {json.dumps(result['error'])}")
    tool_result = result.get("result")
    if not isinstance(tool_result, dict):
        raise SystemExit("GATEWAY_FAILED: response did not contain a JSON-RPC result object")
    if tool_result.get("isError") is True:
        raise SystemExit(f"GATEWAY_FAILED: tool returned isError=true: {json.dumps(tool_result)}")
    serialized = json.dumps(tool_result, sort_keys=True)
    if ORDER_ID.lower() not in serialized.lower():
        raise SystemExit(f"GATEWAY_FAILED: response did not contain expected order ID {ORDER_ID}")
    if EXPECTED_ORDER_STATUS.lower() not in serialized.lower():
        raise SystemExit(f"GATEWAY_FAILED: response did not contain expected status {EXPECTED_ORDER_STATUS}")
    print("GATEWAY_OK")
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


## Execute `verify_memory.py`

Run the cell below.

In [ ]:
"""
Lab 05 - Verify Runtime invocation and AgentCore Memory continuity.

The supplied Runtime must explicitly write and read AgentCore Memory using the
actorId and memorySessionId fields in the payload.
"""
import json
import os
import time
import uuid

import boto3

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
AGENT_RUNTIME_ARN = os.environ.get("AGENT_RUNTIME_ARN", "")
MEMORY_ID = os.environ.get("MEMORY_ID", "")
ACTOR_ID = os.environ.get("ACTOR_ID", "")
MEMORY_SESSION_ID = os.environ.get("MEMORY_SESSION_ID", f"memory-{uuid.uuid4()}")
EXPECTED_TERM = os.environ.get("EXPECTED_TERM", "00001042")
ORDER_ID = os.environ.get("ORDER_ID", "ORD-1001")
EXPECTED_ORDER_STATUS = os.environ.get("EXPECTED_ORDER_STATUS", "SHIPPED")


def require_configuration():
    required = {
        "AGENT_RUNTIME_ARN": AGENT_RUNTIME_ARN,
        "MEMORY_ID": MEMORY_ID,
        "ACTOR_ID": ACTOR_ID,
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise SystemExit(f"CONFIG_MISSING: export {', '.join(missing)}")


def read_response(response):
    content_type = response.get("contentType", "")
    body = response["response"]
    if "text/event-stream" in content_type:
        chunks = []
        for raw_line in body.iter_lines(chunk_size=10):
            line = raw_line.decode("utf-8").strip()
            if line.startswith("data: "):
                chunks.append(line[6:])
        return "\n".join(chunks)
    raw = body.read().decode("utf-8")
    if content_type.startswith("application/json"):
        return json.dumps(json.loads(raw), sort_keys=True)
    return raw


def invoke(agentcore, prompt: str, runtime_session_id: str):
    payload = {
        "prompt": prompt,
        "actorId": ACTOR_ID,
        "memorySessionId": MEMORY_SESSION_ID,
    }
    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        runtimeSessionId=runtime_session_id,
        runtimeUserId=ACTOR_ID,
        contentType="application/json",
        accept="application/json, text/event-stream",
        payload=json.dumps(payload).encode("utf-8"),
    )
    text = read_response(response)
    request_id = response.get("ResponseMetadata", {}).get("RequestId", "unknown")
    print(f"RUNTIME_OK: request_id={request_id} runtime_session_id={runtime_session_id}")
    print(text)
    return text


def wait_for_memory_events(agentcore):
    deadline = time.time() + 30
    while time.time() < deadline:
        response = agentcore.list_events(
            memoryId=MEMORY_ID,
            actorId=ACTOR_ID,
            sessionId=MEMORY_SESSION_ID,
            includePayloads=True,
            maxResults=100,
        )
        events = response.get("events", [])
        if len(events) >= 2:
            return events
        time.sleep(3)
    raise SystemExit("MEMORY_FAILED: fewer than two events were visible after 30 seconds")


def main():
    require_configuration()
    agentcore = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
    first_runtime_session = f"runtime-{uuid.uuid4()}"
    second_runtime_session = f"runtime-{uuid.uuid4()}"

    first_response = invoke(
        agentcore,
        f"Check retail order {ORDER_ID} with the read-only tool and remember that Salesforce case {EXPECTED_TERM} concerns partner portal access.",
        first_runtime_session,
    )
    if ORDER_ID.lower() not in first_response.lower() or EXPECTED_ORDER_STATUS.lower() not in first_response.lower():
        raise SystemExit("TOOL_FAILED: Runtime response did not contain the expected order and status")
    print("TOOL_OK: Runtime invoked the expected read-only retail tool")
    second_response = invoke(
        agentcore,
        "Which Salesforce case did I ask you to remember?",
        second_runtime_session,
    )
    if EXPECTED_TERM not in second_response:
        raise SystemExit(f"CONTINUITY_FAILED: second response did not contain {EXPECTED_TERM}")

    events = wait_for_memory_events(agentcore)
    print(f"MEMORY_OK: actor_id={ACTOR_ID} memory_session_id={MEMORY_SESSION_ID} events={len(events)}")
    print("CONTINUITY_OK: different Runtime sessions retrieved the same Memory session")
    print("Next: use the printed Runtime session IDs to find the corresponding CloudWatch traces.")


if __name__ == "__main__":
    main()


In [ ]:
print("Lab complete.")
print("Takeaway: CloudWatch traces show Runtime status, Gateway tool activity, and Memory context.")